In [157]:
import pandas as pd
import json
import os
from tqdm import tqdm
tqdm.pandas()

task = 'gcd'
final_df = []
for method in os.listdir(f"results/{task}"):
    result_file = f"results/{task}/{method}/results.csv"
    df =pd.read_csv(result_file)
    df = df[~df['args'].isna()]
    df['args'] = df['args'].progress_apply(json.loads)
    df['fold_idx'] = df['args'].apply(lambda x: int(x['fold_idx']))
    df['num_train_epochs'] = df['args'].apply(lambda x: x['num_train_epochs'])
    # df['num_pretrain_epochs'] = df['args'].apply(lambda x: x['num_pretrain_epochs'])
    df = df[(df['num_train_epochs'].apply(int)==50)]
    df = df.drop(['cluster_num_factor', 'args', 'seed', 'num_train_epochs'], axis=1)
    df = df.drop_duplicates(['labeled_ratio', 'known_cls_ratio', 'dataset', 'method', 'fold_idx'])
    final_df.append(df)

final_df = pd.concat(final_df)
final_df = final_df.set_index(['labeled_ratio', 'known_cls_ratio', 'dataset', 'method', 'fold_idx'])
df_melted = final_df.stack().reset_index()
df_melted = df_melted.rename(columns={0: "value", "level_5": "metric"})
df_melted = df_melted.sort_values(['dataset', 'method', 'fold_idx', 'labeled_ratio', 'known_cls_ratio', 'metric'])
df_melted

100%|██████████| 606/606 [00:00<00:00, 50757.81it/s]


100%|██████████| 602/602 [00:00<00:00, 91039.16it/s]


,labeled_ratio,known_cls_ratio,dataset,method,fold_idx,metric,value
11430,0.1,0.25,20NG,ALUP,0,ACC,94.35
11434,0.1,0.25,20NG,ALUP,0,ARI,90.45
11431,0.1,0.25,20NG,ALUP,0,H-Score,96.14
11432,0.1,0.25,20NG,ALUP,0,K-ACC,100.00
11433,0.1,0.25,20NG,ALUP,0,N-ACC,92.56
...,...,...,...,...,...,...,...
11002,0.1,0.75,thucnews,TAN,0,ARI,71.66
10999,0.1,0.75,thucnews,TAN,0,H-Score,75.65
11000,0.1,0.75,thucnews,TAN,0,K-ACC,84.03
11001,0.1,0.75,thucnews,TAN,0,N-ACC,68.80


In [158]:
df_pivot = df_melted.pivot(index=['dataset', 'method', 'fold_idx'], columns=['labeled_ratio', 'known_cls_ratio', 'metric'], values='value')
df_pivot.to_excel(f'results/{task}_pivot.xlsx')
df_pivot

labeled_ratio                0.1                                               \
known_cls_ratio             0.25                                         0.50   
metric                       ACC    ARI H-Score   K-ACC  N-ACC    NMI     ACC   
dataset  method  fold_idx                                                       
20NG     ALUP    0         94.35  90.45   96.14  100.00  92.56  93.46   98.99   
                 1         94.28  95.19   96.04  100.00  92.38  98.26   99.87   
                 2         99.80  99.54   99.77   99.71  99.83  99.63  100.00   
                 3         94.75  91.24   96.12   99.21  93.22  94.52   99.53   
                 4         93.61  89.34   95.35   99.75  91.32  92.92     NaN   
...                          ...    ...     ...     ...    ...    ...     ...   
thucnews PLM_GCD 1         28.64  13.07   29.99   35.89  25.76  24.93   63.76   
                 2         26.60   9.18   26.31   25.69  26.96  19.38   70.51   
                 3         27.52  10.27   28.69   33.08  25.32  21.50   69.25   
         SDC     3           NaN    NaN     NaN     NaN    NaN    NaN   82.47   
         TAN     0         69.54  54.88   72.97   92.89  60.08  67.96   75.69   

labeled_ratio                                      ...     1.0                 \
known_cls_ratio                                    ...    0.50                  
metric                        ARI H-Score   K-ACC  ... H-Score   K-ACC  N-ACC   
dataset  method  fold_idx                          ...                          
20NG     ALUP    0          98.05   99.02   99.57  ...   99.87  100.00  99.74   
                 1          99.69   99.87   99.86  ...   99.94  100.00  99.87   
                 2         100.00  100.00  100.00  ...   99.93  100.00  99.87   
                 3          99.06   99.52   99.87  ...   99.60   99.61  99.59   
                 4            NaN     NaN     NaN  ...     NaN     NaN    NaN   
...                           ...     ...     ...  ...     ...     ...    ...   
thucnews PLM_GCD 1          51.85   61.83   74.82  ...   57.50   67.65  50.00   
                 2          54.96   70.16   66.05  ...   64.33   64.90  63.76   
                 3          54.57   68.21   60.76  ...   66.58   61.32  72.82   
         SDC     3          67.31   82.27   78.01  ...   87.36   84.73  90.16   
         TAN     0          64.98   71.11   94.34  ...     NaN     NaN    NaN   

labeled_ratio                                                             \
known_cls_ratio                     0.75                                   
metric                       NMI     ACC     ARI H-Score   K-ACC   N-ACC   
dataset  method  fold_idx                                                  
20NG     ALUP    0         99.76   99.93   99.85   99.88  100.00   99.76   
                 1         99.88  100.00  100.00  100.00  100.00  100.00   
                 2         99.88  100.00  100.00  100.00  100.00  100.00   
                 3         99.40   99.87   99.68   99.70  100.00   99.40   
                 4           NaN     NaN     NaN     NaN     NaN     NaN   
...                          ...     ...     ...     ...     ...     ...   
thucnews PLM_GCD 1         61.07   60.45   50.69   52.59   67.52   43.07   
                 2         66.00   61.93   50.38   60.61   63.54   57.95   
                 3         64.88   60.24   48.10   63.23   53.29   77.72   
         SDC     3         82.15   93.36   86.20   93.12   93.67   92.57   
         TAN     0           NaN     NaN     NaN     NaN     NaN     NaN   

labeled_ratio                      
known_cls_ratio                    
metric                        NMI  
dataset  method  fold_idx          
20NG     ALUP    0          99.88  
                 1         100.00  
                 2         100.00  
                 3          99.79  
                 4            NaN  
...                           ...  
thucnews PLM_GCD 1          63.10  
                 2          63.

In [159]:
# 基于 df_melted = [labeled_ratio, known_cls_ratio, dataset, method, fold_idx, metric, value]

# 认为：只要某个 metric 有记录，就视为该 (dataset, method, labeled_ratio, known_cls_ratio, fold_idx) 已测试
cells = (df_melted[["dataset","method","labeled_ratio","known_cls_ratio","fold_idx"]]
         .drop_duplicates()
         .sort_values(["dataset","method","labeled_ratio","known_cls_ratio","fold_idx"])
         .reset_index(drop=True))

# 1) 每个 dataset × method 的维度覆盖与计数
progress_by_dm = (cells
    .groupby(["dataset","method"])
    .agg(
        labeled_ratios=("labeled_ratio", lambda s: sorted(pd.unique(s))),
        n_labeled=("labeled_ratio", pd.Series.nunique),
        known_cls_ratios=("known_cls_ratio", lambda s: sorted(pd.unique(s))),
        n_known=("known_cls_ratio", pd.Series.nunique),
        folds=("fold_idx", lambda s: sorted(pd.unique(s))),
        n_folds=("fold_idx", pd.Series.nunique),
        n_cells=("fold_idx", "size"),  # 完成的组合总数
    )
    .reset_index()
)

# 2) 每个 dataset × method × labeled_ratio × known_cls_ratio 下完成了多少个 fold
fold_coverage = (cells
    .groupby(["dataset","method","labeled_ratio","known_cls_ratio"])["fold_idx"]
    .nunique()
    .reset_index(name="folds_done")
    .sort_values(["dataset","method","labeled_ratio","known_cls_ratio"])
)

In [160]:
# ====== 2) （可选）若有计划 fold 列表，显示“完成数/计划数” ======
# 例如计划 folds 为 [0,1,2,3,4]（共5个）
EXPECTED_FOLDS = None  # e.g., [0,1,2,3,4]
if EXPECTED_FOLDS is not None:
    plan_n = len(EXPECTED_FOLDS)
    # 统计每格完成的 fold 个数
    fold_done_cnt = fold_coverage.copy()
    fold_done_cnt["plan"] = plan_n
    fold_done_cnt["text"] = fold_done_cnt["folds_done"].astype(int).astype(str) + "/" + str(plan_n)
    pivot_folds_text = fold_done_cnt.pivot(
        index=["dataset", "method"],
        columns=["labeled_ratio", "known_cls_ratio"],
        values="text"
    ).sort_index()

# ====== 3) 每格显示“已完成的 fold 列表”，便于直观看缺哪几个 ======
# 先做一个按格聚合出 fold 列表的表
fold_list = (cells.groupby(["dataset","method","labeled_ratio","known_cls_ratio"])["fold_idx"]
             .apply(lambda s: sorted(pd.unique(s)))
             .reset_index(name="folds_list"))
fold_list["folds_list_str"] = fold_list["folds_list"].apply(lambda x: "[" + ", ".join(map(str, x)) + "]")

pivot_folds_list = fold_list.pivot(
    index=["dataset", "method"],
    columns=["labeled_ratio", "known_cls_ratio"],
    values="folds_list_str"
).sort_index()

pivot_folds_list.to_excel(f'results/{task}_progress.xlsx')
pivot_folds_list

labeled_ratio                     0.1                              \
known_cls_ratio                  0.25          0.50          0.75   
dataset  method                                                     
20NG     ALUP         [0, 1, 2, 3, 4]  [0, 1, 2, 3]  [0, 1, 2, 3]   
         DPN          [0, 1, 2, 3, 4]  [0, 1, 2, 3]  [0, 1, 2, 3]   
         DeepAligned  [0, 1, 2, 3, 4]  [0, 1, 2, 3]  [0, 1, 2, 3]   
         GeoID           [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]   
         LOOP         [0, 1, 2, 3, 4]  [0, 1, 2, 3]  [0, 1, 2, 3]   
...                               ...           ...           ...   
thucnews ALUP            [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]   
         DPN             [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]   
         PLM_GCD         [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]   
         SDC                      NaN           [3]           [3]   
         TAN                      [0]           [0]           [0]   

labeled_ratio                  0.5                                       1.0  \
known_cls_ratio               0.25          0.50          0.75          0.25   
dataset  method                                                                
20NG     ALUP         [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]   
         DPN          [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]   
         DeepAligned  [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]   
         GeoID        [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]   
         LOOP         [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]   
...                            ...           ...           ...           ...   
thucnews ALUP         [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]   
         DPN          [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]   
         PLM_GCD      [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]   
         SDC                   [3]           [3]           [3]           [3]   
         TAN                   NaN           NaN           NaN           NaN   

labeled_ratio                                     
known_cls_ratio               0.50          0.75  
dataset  method                                   
20NG     ALUP         [0, 1, 2, 3]  [0, 1, 2, 3]  
         DPN          [0, 1, 2, 3]  [0, 1, 2, 3]  
         DeepAligned  [0, 1, 2, 3]  [0, 1, 2, 3]  
         GeoID        [0, 1, 2, 3]  [0, 1, 2, 3]  
         LOOP         [0, 1, 2, 3]  [0, 1, 2, 3]  
...                            ...           ...  
thucnews ALUP         [0, 1, 2, 3]  [0, 1, 2, 3]  
         DPN          [0, 1, 2, 3]  [0, 1, 2, 3]  
         PLM_GCD      [0, 1, 2, 3]  [0, 1, 2, 3]  
         SDC                   [3]           [3]  
         TAN                   NaN           NaN  

[86 rows x 9 columns]

In [161]:
all_num = df_melted['dataset'].nunique() * 3 * 3 * df_melted['method'].nunique() * 5
cur_num = len(df_melted) / df_melted['metric'].nunique()
cur_num / all_num

0.5851851851851851

In [162]:
all_num

4455

In [163]:
cur_num

2607.0